# Build RAG


## Paths


In [34]:
# paths

from pathlib import Path


cwd = Path.cwd().resolve()

candidates = []

for path in [cwd, *cwd.parents]:
    candidates.extend([
        path,
        path / "fitness-assistant",
        path / "07-project-example" / "fitness-assistant",
        path / "datatalks" / "llm" / "zoomcamp-llm-2026" / "07-project-example" / "fitness-assistant",
    ])

PROJECT_DIR = None

for candidate in candidates:
    if (candidate / "data" / "data.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find fitness-assistant/data/data.csv")

COURSE_ROOT = None

for path in [PROJECT_DIR, *PROJECT_DIR.parents]:
    if (path / "pyproject.toml").exists():
        COURSE_ROOT = path
        break

if COURSE_ROOT is None:
    raise FileNotFoundError("Could not find course root with pyproject.toml")

DATA_DIR = PROJECT_DIR / "data"

print("Project dir:", PROJECT_DIR)
print("Data dir:", DATA_DIR)


Project dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant
Data dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/07-project-example/fitness-assistant/data


## Packages


In [35]:
# packages and client

import os

import pandas as pd
from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI


load_dotenv(COURSE_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to the course root .env file.")

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

MODEL


'gpt-5.4-mini'

## Load data


In [36]:
# load csv

csv_path = DATA_DIR / "data.csv"

df = pd.read_csv(csv_path)
documents = df.to_dict(orient="records")

print("Rows:", len(df))
df.head()


Rows: 50


,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a high plank with hands slightly wide...
1,squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with feet shoulder-width apart and toes ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Shoulders, Quadriceps, Calves, Glutes, Core",Stand upright with feet together and arms at y...


## Index


In [37]:
# minsearch index

index = Index(
    text_fields=[
        "exercise_name",
        "type_of_activity",
        "type_of_equipment",
        "body_part",
        "type",
        "muscle_groups_activated",
        "instructions",
    ],
    keyword_fields=["id"],
)

index.fit(documents)


## Search


In [38]:
# search function

def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10,
    )

    return results


## Test search


In [39]:
# quick search test

query = "Give me leg exercises for hamstrings"
results = search(query)

results[0]


{'id': 'single-leg-romanian-deadlift-046',
 'exercise_name': 'Single-Leg Romanian Deadlift',
 'type_of_activity': 'Strength',
 'type_of_equipment': 'Dumbbells',
 'body_part': 'Hamstrings',
 'type': 'Compound',
 'muscle_groups_activated': 'Hamstrings, Glutes, Core, Balance Muscles',
 'instructions': 'Stand on one leg holding dumbbells or a single weight. Hinge at the hips while extending the free leg behind you and lowering the weight toward the floor. Keep your back flat and hips square. Return to standing by driving through the standing heel and squeezing the glute.'}

## Prompt


In [40]:
# prompt templates

prompt_template = """
You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}
""".strip()


## Build prompt


In [41]:
# build prompt

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt


## Check prompt


In [42]:
# inspect prompt

prompt = build_prompt(query, results)
print(prompt[:1500])


You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: Give me leg exercises for hamstrings

CONTEXT:
exercise_name: Single-Leg Romanian Deadlift
type_of_activity: Strength
type_of_equipment: Dumbbells
body_part: Hamstrings
type: Compound
muscle_groups_activated: Hamstrings, Glutes, Core, Balance Muscles
instructions: Stand on one leg holding dumbbells or a single weight. Hinge at the hips while extending the free leg behind you and lowering the weight toward the floor. Keep your back flat and hips square. Return to standing by driving through the standing heel and squeezing the glute.

exercise_name: Lying Hamstring Curl
type_of_activity: Strength
type_of_equipment: Machine
body_part: Hamstrings
type: Isolation
muscle_groups_activated: Hamstrings, Calves
instructions: Lie face down on the hamstring curl machine and position the pad just above your ankles. Bend your k

## LLM


In [43]:
# model call

def llm(prompt, model=MODEL):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
    )

    return response.output_text


## RAG


In [44]:
# rag function

def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)
    return answer


## Test RAG


In [45]:
# first answer

question = "Is the Lat Pulldown considered a strength training activity, and if so, why?"
answer = rag(question)

print(answer)


Yes. The Lat Pulldown is considered a **strength** activity because the context lists its **type_of_activity** as **Strength**. It is also a **compound** machine exercise that activates the **latissimus dorsi, biceps, rhomboids, and rear deltoids**.


## Another question


In [46]:
# second answer

question = "Give me leg exercises for hamstrings"
answer = rag(question)

print(answer)


Here are some leg exercises for hamstrings from the database:

- Single-Leg Romanian Deadlift
- Lying Hamstring Curl
- Romanian Deadlift
- Leg Press
- Glute Bridge
- Barbell Hip Thrust
- Kettlebell Swing
- Forward Lunge
- Step-Up

If you want, I can also sort these by equipment or whether they’re compound or isolation exercises.
